# 02 – Does order book imbalance predict the next price move?

**Question:** when there is more size on the bid than the ask, is the next mid-price move more likely to be up?

**Method:**
1. Measure imbalance at the best level and across the top 3 levels.
2. Look at the probability of an up move for each imbalance bucket, with 95% confidence intervals.
3. Test out of sample: train on earlier days, test on the last day, sampling once per second so the test rows are not near-duplicates.
4. Check whether the microprice predicts the mid price 1 second ahead better than the mid itself.

Run `rebuild_book.py` or `rebuild_tardis.py` first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

TICK = 0.01        # BTCUSDT tick size
LV = 3             # levels used for the multi-level imbalance
HORIZON_MS = 1000  # horizon for the microprice test

cols = ["event_ms"] + [f"{s}_{k}_{i}" for i in range(LV) for s in ("bid", "ask") for k in ("px", "qty")]
book = pd.read_parquet("data/processed/book", columns=cols)  # only the columns we need, to save memory
book = book.sort_values("event_ms", kind="stable").reset_index(drop=True)
book["time"] = pd.to_datetime(book.event_ms, unit="ms")
book["day"] = book.time.dt.date
print(f"{len(book):,} book updates over {book.day.nunique()} day(s)")

## Features

In [ ]:
b0, a0 = book.bid_qty_0, book.ask_qty_0
book["mid"] = (book.bid_px_0 + book.ask_px_0) / 2
book["spread_ticks"] = ((book.ask_px_0 - book.bid_px_0) / TICK).round()
book["imb1"] = (b0 - a0) / (b0 + a0)                      # best level only
bsum = sum(book[f"bid_qty_{i}"] for i in range(LV))
asum = sum(book[f"ask_qty_{i}"] for i in range(LV))
book["imb3"] = (bsum - asum) / (bsum + asum)              # top 3 levels
book["microprice"] = (book.bid_px_0 * a0 + book.ask_px_0 * b0) / (b0 + a0)

## Targets

- `next_move`: direction (+1 or −1) of the next change in the mid price, however long it takes.
- `future_mid`: the mid price `HORIZON_MS` later.

Computed separately for each day so nothing leaks across days.

In [ ]:
def add_targets(g):
    g = g.copy()
    mid = g.mid.to_numpy()
    n = len(mid)
    chg = np.diff(mid)                         # chg[j] = mid[j+1] - mid[j]
    nz = np.flatnonzero(chg != 0)              # updates where the mid moved
    pos = np.searchsorted(nz, np.arange(n))    # first move at or after each row
    nxt = np.full(n, np.nan)
    ok = pos < len(nz)
    nxt[ok] = np.sign(chg[nz[pos[ok]]])
    g["next_move"] = nxt

    left = pd.DataFrame({"t": g.event_ms.to_numpy() + HORIZON_MS})
    right = pd.DataFrame({"t": g.event_ms.to_numpy(), "future_mid": mid})
    g["future_mid"] = pd.merge_asof(left, right, on="t", direction="backward").future_mid.to_numpy()
    g.loc[g.event_ms > g.event_ms.max() - HORIZON_MS, "future_mid"] = np.nan  # no future at end of day
    return g

book = pd.concat([add_targets(g) for _, g in book.groupby("day")], ignore_index=True)

# Compare like with like: keep only the most common spread (usually 1 tick).
common_spread = book.spread_ticks.mode()[0]
print(f"most common spread: {common_spread:.0f} tick(s), {(book.spread_ticks == common_spread).mean():.1%} of updates")
d = book[(book.spread_ticks == common_spread) & book.next_move.notna()].copy()

## 1. Probability of an up move by imbalance bucket

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

d["bucket"] = pd.cut(d.imb1, np.linspace(-1, 1, 11), include_lowest=True)
tab = d.groupby("bucket", observed=True).next_move.agg(n="size", ups=lambda s: int((s > 0).sum()))
tab["p_up"] = tab.ups / tab.n
tab["ci_low"], tab["ci_high"] = wilson(tab.ups, tab.n)
print(tab.round(3))

centres = [b.mid for b in tab.index]
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(centres, tab.p_up, yerr=[tab.p_up - tab.ci_low, tab.ci_high - tab.p_up], fmt="o", capsize=3)
ax.axhline(0.5, ls="--", c="grey")
ax.set_xlabel("imbalance at best level"); ax.set_ylabel("P(next mid move is up)")
ax.set_title("Next mid move vs. order book imbalance (95% CI)"); plt.show()

## 2. Out-of-sample test

Consecutive book updates are almost identical, so testing on every row overstates the evidence.
We keep one row per second and train on earlier days, testing on the last day
(or on the last 30% of the data if there is only one day).

In [ ]:
s = d.assign(sec=d.event_ms // 1000).groupby("sec").head(1).copy()
s["up"] = (s.next_move > 0).astype(int)

days = sorted(s.day.unique())
if len(days) > 1:
    train, test = s[s.day < days[-1]], s[s.day == days[-1]]
    print(f"train: {days[:-1]}  test: {days[-1]}")
else:
    cut = s.event_ms.quantile(0.7)
    train, test = s[s.event_ms <= cut], s[s.event_ms > cut]
    print("single day: train on first 70%, test on last 30%")
print(f"{len(train):,} train rows, {len(test):,} test rows")

base = train.up.mean()
results = {"base rate (no signal)": {
    "accuracy": accuracy_score(test.up, np.full(len(test), int(base >= 0.5))),
    "log_loss": log_loss(test.up, np.full(len(test), base), labels=[0, 1]),
    "auc": 0.5}}
for name, feats in {"imbalance, best level": ["imb1"],
                    "imbalance, top 3 levels": ["imb3"],
                    "both": ["imb1", "imb3"]}.items():
    model = LogisticRegression().fit(train[feats], train.up)
    p = model.predict_proba(test[feats])[:, 1]
    results[name] = {"accuracy": accuracy_score(test.up, p >= 0.5),
                     "log_loss": log_loss(test.up, p, labels=[0, 1]),
                     "auc": roc_auc_score(test.up, p)}
pd.DataFrame(results).T.round(4)

## 3. Microprice vs. mid as a predictor of the price 1 second ahead

Lower error means a better estimate of where the price is going.

In [ ]:
t = test[test.future_mid.notna()]
rmse = lambda e: np.sqrt(np.mean(e**2)) / TICK
err_mid = rmse(t.future_mid - t.mid)
err_micro = rmse(t.future_mid - t.microprice)
print(f"RMSE using mid:        {err_mid:.4f} ticks")
print(f"RMSE using microprice: {err_micro:.4f} ticks")
print(f"improvement: {1 - err_micro / err_mid:.1%}")

## Notes for the README

- Does P(up) rise steadily with imbalance, and do the confidence intervals exclude 0.5 at the extremes?
- How much better than the base rate is the model out of sample (AUC, log loss)?
- Is the top-3 imbalance more informative than the best level alone?
- How much does the microprice improve on the mid?
- Limitations: one exchange, one symbol, a few days; the next move can take a long time to arrive.